In [ ]:
import sys
import pandas as pd 
import matplotlib.pyplot as plt 
import numpy as np
from IPython.display import display

from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))


In [ ]:
from data_processing.constants import DEFAULT_TICKERS
from data_analysis.loader import MarketDataLoader
from data_analysis.cleaner import MarketDataCleaner
from data_analysis.features import MarketFeatureEngineer

loader = MarketDataLoader()
raw_df = loader.load_analysis_data(DEFAULT_TICKERS)

print("Raw Shape:", raw_df.shape)
clean_df = MarketDataCleaner.clean(raw_df)
df = MarketFeatureEngineer.add_all_features(clean_df)
print("Cleaned and Engineered Shape:", df.shape)


ModuleNotFoundError: No module named 'backend'

In [ ]:
from data_analysis.analysis import (
    instrument_summary,
    risk_return_summary,
    asset_class_summary,
    return_distribution_analysis,
    drawdown_summary,
    correlation_matrix,
    asset_class_correlation, 
    top_bottom_performers,
    monthly_performance,
    annual_performance,
    data_completeness_check
)


## Instrument Summary

In [5]:
# check metadata
df.columns.tolist()
metadata = loader.load_instrument_metadata()
metadata.head()
metadata.shape


2026-09-15 08:53:21,597 - ERROR - Failed to load instrument metadata: connection to server at "10.9.69.224", port 5432 failed: Connection timed out
	Is the server running on that host and accepting TCP/IP connections?



(0, 0)

In [ ]:
instrument_stats = instrument_summary(df)
completeness = data_completeness_check(df)

display(
    instrument_stats[[
        "symbol", "name", "asset_class", "observations",
        "daily_mean_return", "daily_std_return",
        "final_cumulative_return", "max_drawdown",
        "data_completeness_pct"
    ]].sort_values(by="final_cumulative_return", ascending=False).head(10)
)

display(
    completeness.sort_values(by="completeness_pct").head(10)
)

## Daily return distribution

In [ ]:
return_dist = return_distribution_analysis(df)
display(
    return_dist.sort_values(by="std", ascending=False).head(10)
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(df["daily_return"].dropna(), bins=80, color="steelblue", edgecolor="white")
ax.set_title("Distribution of Daily Returns")
ax.set_xlabel("Daily Return")
ax.set_ylabel("Frequency")
ax.axvline(df["daily_return"].mean(), color="crimson", linestyle="--", label="Mean Return")
ax.legend()
fig.tight_layout()
plt.show()

## Return Distribution by instrument

In [ ]:
selected_symbols = instrument_stats.sort_values(by="observations", ascending=False)["symbol"].head(8).tolist()
symbol_returns = [
    df.loc[df["symbol"] == symbol, "daily_return"].dropna()
    for symbol in selected_symbols
]

fig, ax = plt.subplots(figsize=(12, 6))
ax.boxplot(symbol_returns, labels=selected_symbols, showfliers=False)
ax.set_title("Daily Return Distribution for Selected Instruments")
ax.set_xlabel("Symbol")
ax.set_ylabel("Daily Return")
plt.xticks(rotation=45)
fig.tight_layout()
plt.show()

## Risk and return

In [ ]:
risk_return = risk_return_summary(df)
asset_class_risk = asset_class_summary(risk_return)

display(risk_return.sort_values(by="annual_return", ascending=False).head(10))
display(asset_class_risk.sort_values(by="avg_annual_return", ascending=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for asset_class_name, subset in risk_return.groupby("asset_class"):
    ax.scatter(
        subset["annual_volatility"],
        subset["annual_return"],
        label=asset_class_name,
        alpha=0.7
    )

ax.set_title("Risk vs Return by Instrument")
ax.set_xlabel("Annual Volatility")
ax.set_ylabel("Annual Return")
ax.axhline(0, color="grey", linewidth=1)
ax.legend(title="Asset Class")
fig.tight_layout()
plt.show()

## Top performers

In [ ]:
performers = top_bottom_performers(df, n=10)
top_performers = performers["top"].copy()
display(top_performers)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_performers["symbol"], top_performers["cumulative_return"], color="seagreen")
ax.set_title("Top 10 Performers by Cumulative Return")
ax.set_xlabel("Cumulative Return")
ax.set_ylabel("Symbol")
fig.tight_layout()
plt.show()

## Lowest Perfromers

In [ ]:
bottom_performers = performers["bottom"].copy()
display(bottom_performers)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(bottom_performers["symbol"], bottom_performers["cumulative_return"], color="indianred")
ax.set_title("Bottom 10 Performers by Cumulative Return")
ax.set_xlabel("Cumulative Return")
ax.set_ylabel("Symbol")
fig.tight_layout()
plt.show()

## Maximum Drawdown

In [ ]:
drawdown_stats = drawdown_summary(df)
display(drawdown_stats.sort_values(by="max_drawdown_pct").head(10))

fig, ax = plt.subplots(figsize=(10, 6))
worst_drawdowns = drawdown_stats.sort_values(by="max_drawdown_pct").head(10)
ax.barh(worst_drawdowns["symbol"], worst_drawdowns["max_drawdown_pct"], color="darkorange")
ax.set_title("Worst Maximum Drawdowns")
ax.set_xlabel("Maximum Drawdown (%)")
ax.set_ylabel("Symbol")
fig.tight_layout()
plt.show()

## Correlation Matrix

In [ ]:
corr = correlation_matrix(df)
display(corr.round(2).iloc[:10, :10])

fig, ax = plt.subplots(figsize=(12, 8))
image = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_title("Correlation Matrix of Daily Returns")
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(corr.index)))
ax.set_yticklabels(corr.index, fontsize=8)
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
plt.show()

## Asset Class Correlation

In [ ]:
asset_corr = asset_class_correlation(df)
display(asset_corr.sort_values(by="avg_correlation", ascending=False))

## Monthly Performance

In [ ]:
monthly = monthly_performance(df)
monthly_summary = (
    monthly.groupby("symbol")["monthly_return"]
    .agg(avg_monthly_return="mean", median_monthly_return="median")
    .reset_index()
    .sort_values(by="avg_monthly_return", ascending=False)
)
display(monthly_summary.head(10))

monthly_asset = (
    monthly.groupby([monthly["year_month"].astype(str), "asset_class"])["monthly_return"]
    .mean()
    .unstack()
)
fig, ax = plt.subplots(figsize=(12, 6))
for asset_class_name in monthly_asset.columns:
    ax.plot(monthly_asset.index, monthly_asset[asset_class_name], label=asset_class_name)
ax.set_title("Average Monthly Return by Asset Class")
ax.set_xlabel("Month")
ax.set_ylabel("Average Monthly Return")
ax.legend(title="Asset Class")
plt.xticks(rotation=90)
fig.tight_layout()
plt.show()

## Annual Performance

In [ ]:
annual = annual_performance(df)
annual_summary = (
    annual.groupby(["symbol", "asset_class"])["annual_return"]
    .mean()
    .reset_index()
    .sort_values(by="annual_return", ascending=False)
)
display(annual_summary.head(10))

annual_asset = annual.groupby(["year", "asset_class"])["annual_return"].mean().unstack()
fig, ax = plt.subplots(figsize=(10, 6))
for asset_class_name in annual_asset.columns:
    ax.plot(annual_asset.index, annual_asset[asset_class_name], marker="o", label=asset_class_name)
ax.set_title("Average Annual Return by Asset Class")
ax.set_xlabel("Year")
ax.set_ylabel("Average Annual Return")
ax.legend(title="Asset Class")
fig.tight_layout()
plt.show()